Use a double dueling DQN to train an agent that can achieve a
superhuman level at the famous Atari Breakout game ("ALE/Breakout-
v5"). The observations are images. To simplify the task, you should
convert them to grayscale (i.e., average over the channels axis) then crop
them and downsample them, so they’re just large enough to play, but not
more. An individual image does not tell you which way the ball and the
paddles are going, so you should merge two or three consecutive images
to form each state. Lastly, the DQN should be composed mostly of
convolutional layers.

In [12]:
%pip install "gymnasium[atari]" ale-py
%pip install autorom AutoROM

In [1]:
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)
env = gym.make("ALE/Breakout-v5", render_mode="rgb_array")

obs, info = env.reset()
obs.shape

(210, 160, 3)

In [2]:
import tensorflow as tf
import numpy as np
from collections import deque

FRAME_STACK_SIZE = 3
PREPROCESSED_HEIGHT = 84
PREPROCESSED_WIDTH = 84

def preprocess_observation(observation):
    grayscale = np.mean(observation, axis=2).astype(np.float32)
    cropped = grayscale[34:194, :]
    resized = tf.image.resize(cropped[..., np.newaxis], [PREPROCESSED_HEIGHT, PREPROCESSED_WIDTH])
    return resized.numpy()[:, :, 0] / 255.0

batch_size = 32
discount_factor = 0.99
optimizer = tf.keras.optimizers.Nadam(learning_rate=0.00025)
loss_fn = tf.keras.losses.MeanSquaredError()
input_shape = (PREPROCESSED_HEIGHT, PREPROCESSED_WIDTH, FRAME_STACK_SIZE)
n_outputs = 4

inputs = tf.keras.Input(shape=input_shape)

x = tf.keras.layers.Conv2D(32, kernel_size=8, strides=4, activation="relu")(inputs)
x = tf.keras.layers.Conv2D(64, kernel_size=4, strides=2, activation="relu")(x)
x = tf.keras.layers.Conv2D(64, kernel_size=3, strides=1, activation="relu")(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(512, activation="relu")(x)

state_value = tf.keras.layers.Dense(1)(x)
advantage = tf.keras.layers.Dense(n_outputs)(x)
advantage = advantage - tf.keras.ops.mean(advantage, axis=1, keepdims=True)
q_values = state_value + advantage

online_model = tf.keras.Model(inputs=inputs, outputs=q_values)

target_model = tf.keras.models.clone_model(online_model)
target_model.set_weights(online_model.get_weights())

In [ ]:
replay_buffer = deque(maxlen=100_000)

def sample_experiences(batch_size):
    indices = np.random.randint(len(replay_buffer), size=batch_size)
    batch = [replay_buffer[index] for index in indices]
    return [
        np.array([experience[field_index] for experience in batch])
        for field_index in range(6)
    ]  # [states, actions, rewards, next_states, dones, truncateds]

In [4]:
def play_one_step(env, state, epsilon):
    if np.random.rand() < epsilon:
        action = env.action_space.sample()
    else:
        q_values = online_model.predict(state[np.newaxis], verbose=0)[0]
        action = q_values.argmax()

    observation, reward, done, truncated, info = env.step(action)
    next_frame = preprocess_observation(observation)
    next_state = np.concatenate([state[:, :, 1:], next_frame[:, :, np.newaxis]], axis=2)
    replay_buffer.append((state, action, reward, next_state, done, truncated))
    return next_state, reward, done, truncated, info

In [ ]:
def training_step(batch_size):
    experiences = sample_experiences(batch_size)
    states, actions, rewards, next_states, dones, truncateds = experiences

    next_q_values = online_model.predict(next_states, verbose=0)
    best_next_actions = next_q_values.argmax(axis=1)
    next_mask = tf.one_hot(best_next_actions, n_outputs).numpy()

    target_next_q_values = target_model.predict(next_states, verbose=0)
    max_next_q_values = (target_next_q_values * next_mask).sum(axis=1)

    not_done_mask = 1.0 - np.logical_or(dones, truncateds).astype(np.float32)
    target_q_values = rewards + not_done_mask * discount_factor * max_next_q_values
    target_q_values = target_q_values.reshape(-1, 1)

    action_mask = tf.one_hot(actions, n_outputs)

    with tf.GradientTape() as tape:
        all_q_values = online_model(states)
        q_values = tf.reduce_sum(all_q_values * action_mask, axis=1, keepdims=True)
        loss = loss_fn(target_q_values, q_values)

    grads = tape.gradient(loss, online_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, online_model.trainable_variables))

In [6]:
episode_returns = []
episodes = 600
total_steps = 0

epsilon_start = 1.0
epsilon_end = 0.01
epsilon_decay_steps = 200_000
target_update_every_n_steps = 2000
min_replay_buffer_size = 1000
train_every_n_steps = 4

for episode in range(episodes):
    observation, info = env.reset()
    initial_frame = preprocess_observation(observation)
    state = np.stack([initial_frame] * FRAME_STACK_SIZE, axis=2)

    episode_reward = 0
    for step in range(10_000):
        epsilon = max(epsilon_end, epsilon_start - (epsilon_start - epsilon_end) * total_steps / epsilon_decay_steps)
        state, reward, terminated, truncated, info = play_one_step(env, state, epsilon)
        episode_reward += reward
        total_steps += 1

        if len(replay_buffer) >= min_replay_buffer_size and total_steps % train_every_n_steps == 0:
            training_step(batch_size)

        if total_steps % target_update_every_n_steps == 0:
            target_model.set_weights(online_model.get_weights())

        if terminated or truncated:
            break

    episode_returns.append(episode_reward)
    if episode % 50 == 0:
        print(f"Episode {episode}, Reward: {episode_reward:.1f}, Epsilon: {epsilon:.3f}, Steps: {total_steps}")

env.close()

Episode 0, Reward: 3.0, Epsilon: 0.999, Steps: 268
Episode 50, Reward: 2.0, Epsilon: 0.954, Steps: 9329
Episode 100, Reward: 3.0, Epsilon: 0.907, Steps: 18845
Episode 150, Reward: 1.0, Epsilon: 0.860, Steps: 28298
Episode 200, Reward: 0.0, Epsilon: 0.812, Steps: 38058
Episode 250, Reward: 0.0, Epsilon: 0.763, Steps: 47915
Episode 300, Reward: 2.0, Epsilon: 0.705, Steps: 59499
Episode 350, Reward: 3.0, Epsilon: 0.638, Steps: 73123
Episode 400, Reward: 3.0, Epsilon: 0.563, Steps: 88292
Episode 450, Reward: 1.0, Epsilon: 0.491, Steps: 102792
Episode 500, Reward: 7.0, Epsilon: 0.417, Steps: 117698
Episode 550, Reward: 7.0, Epsilon: 0.342, Steps: 133001


In [7]:
test_environment = gym.make("ALE/Breakout-v5", render_mode="rgb_array")
test_episodes = 5
test_rewards = []

for episode in range(test_episodes):
    observation, info = test_environment.reset()
    initial_frame = preprocess_observation(observation)
    state = np.stack([initial_frame] * FRAME_STACK_SIZE, axis=2)
    episode_reward = 0
    done = False

    while not done:
        q_values = online_model.predict(state[np.newaxis], verbose=0)[0]
        action = q_values.argmax()
        observation, reward, terminated, truncated, info = test_environment.step(action)
        next_frame = preprocess_observation(observation)
        state = np.concatenate([state[:, :, 1:], next_frame[:, :, np.newaxis]], axis=2)
        episode_reward += reward
        done = terminated or truncated

    test_rewards.append(episode_reward)
    print(f"Episode {episode + 1}: Total Reward = {episode_reward}")

test_environment.close()
print(f"\nAverage Reward over {test_episodes} episodes: {np.mean(test_rewards):.2f}")

Episode 1: Total Reward = 9.0
Episode 2: Total Reward = 13.0
Episode 3: Total Reward = 19.0
Episode 4: Total Reward = 16.0
Episode 5: Total Reward = 18.0

Average Reward over 5 episodes: 15.00


In [8]:
from google.colab import files

model_filename = "double_dueling_dqn_breakout.keras"
online_model.save(model_filename)
files.download(model_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>